# Decision Trees: Interpretable Machine Learning

A comprehensive exploration of one of the most intuitive and interpretable algorithms in machine learning. Decision trees form the building block for powerful ensemble methods like Random Forests and Gradient Boosting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris, load_wine, make_regression, load_breast_cancer
from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    plot_tree,
    export_graphviz,
    export_text,
)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
print("All imports successful!")

## Intuition: How Decision Trees Work

A decision tree mimics human decision-making by asking a series of **hierarchical questions** about the data.

**Analogy: The "20 Questions" Game**

Imagine you're trying to guess an animal. You ask:
1. "Does it live in water?" → Yes (you move to a subset of aquatic animals)
2. "Does it have fins?" → Yes (you narrow to fish)
3. "Is it bigger than a breadbox?" → No (you guess "goldfish")

Each question splits the set of possibilities into smaller, more homogeneous subsets. That's exactly what a decision tree does!

### Key Idea
- At each node, the tree chooses the **best feature and threshold** to split the data
- "Best" means the split that creates the **purest** child nodes (where each child contains mostly one class)
- This process repeats recursively until a stopping criterion is met

## Anatomy of a Decision Tree

```
                    ┌─────────────────────┐
                    │      ROOT NODE      │
                    │  petal length ≤ 2.45 │
                    └──────────┬──────────┘
                               │
              ┌────────────────┴────────────────┐
              │                                 │
    ┌─────────▼──────────┐           ┌──────────▼───────────┐
    │  DECISION NODE      │           │   LEAF NODE          │
    │  petal width ≤ 1.75 │           │   Class: Setosa      │
    └─────────┬──────────┘           └──────────────────────┘
              │
     ┌────────┴────────┐
     │                 │
┌────▼────┐      ┌─────▼─────┐
│  Leaf   │      │   Leaf    │
│ Versicolor│    │ Virginica │
└─────────┘      └───────────┘
```

### Components
- **Root Node**: The top node; contains the entire dataset and the first split
- **Decision Nodes**: Internal nodes where a feature-based test splits the data further
- **Leaf Nodes**: Terminal nodes that provide the final prediction (class or value)
- **Branches**: The connections between nodes, representing the outcome of a split decision

### How Predictions Flow
A new sample starts at the root, follows branches based on feature comparisons, and lands in a leaf. The leaf's majority class (classification) or mean value (regression) becomes the prediction.

## Splitting Criteria: Measuring Split Quality

The tree must decide which feature and threshold produce the best split. This is measured by impurity metrics:

### 1. Gini Impurity
Measures how often a randomly chosen sample would be misclassified if randomly labeled according to class distribution.

$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

- **Range**: $[0, 0.5]$ for binary classification
- **Minimum (0)**: All samples belong to one class (pure node)
- **Maximum (0.5)**: Samples are equally split (impure)

### 2. Entropy / Information Gain
Based on information theory, entropy measures the disorder in a node.

$$Entropy = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

$$Information\ Gain = Entropy_{parent} - \sum_{k=1}^{K} \frac{N_k}{N} \times Entropy_{child_k}$$

- **Range**: $[0, 1.0]$ for binary classification
- The tree chooses the split that **maximizes Information Gain**

### 3. MSE (for Regression)
For regression trees, the splitting criterion is the Mean Squared Error:

$$MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \bar{y})^2$$

The tree chooses splits that minimize the weighted MSE of child nodes.

### Gini vs Entropy: Practical Differences
- Both usually yield **very similar trees**
- Gini is **slightly faster** computationally (no log calculations)
- Entropy is more **theoretically grounded** in information theory
- Scikit-learn defaults to Gini for its speed advantage

In [ ]:
# Visual comparison of Gini vs Entropy
p = np.linspace(0.001, 0.999, 200)
gini = 2 * p * (1 - p)
entropy = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
entropy_scaled = entropy / 2  # scale to same max as gini

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full curves
axes[0].plot(p, gini, "b-", linewidth=2.5, label="Gini Impurity: $2p(1-p)$")
axes[0].plot(p, entropy, "r-", linewidth=2.5, label="Entropy: $-p\log_2(p) - (1-p)\log_2(1-p)$")
axes[0].plot(p, entropy_scaled, "g--", linewidth=2, label="Entropy / 2 (scaled)")
axes[0].axvline(0.5, color="gray", linestyle=":", alpha=0.5)
axes[0].set_xlabel("Probability of Class 1 ($p$)", fontsize=12)
axes[0].set_ylabel("Impurity", fontsize=12)
axes[0].set_title("Splitting Criteria Comparison", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3)

# Right: zoomed difference
axes[1].plot(p, entropy - gini, "purple", linewidth=2)
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Probability of Class 1 ($p$)", fontsize=12)
axes[1].set_ylabel("Entropy - Gini", fontsize=12)
axes[1].set_title("Difference (Entropy - Gini)", fontsize=14)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Gini at p=0.5: {gini[len(p)//2]:.4f}")
print(f"Entropy at p=0.5: {entropy[len(p)//2]:.4f}")
print(f"Entropy penalizes impure nodes more heavily than Gini.")

## Training a DecisionTreeClassifier with scikit-learn

Let's work with the classic **Iris dataset** — 150 samples of 3 iris species with 4 features each.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
class_names = iris.target_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Classes: {class_names}")
print(f"Class distribution: {np.bincount(y)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

# Train a default Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n{'='*50}")
print(f"Decision Tree Accuracy: {acc:.4f}")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## Visualizing the Decision Tree

One of the biggest advantages of decision trees is **interpretability** — we can literally see and trace every decision.

In [ ]:
# Method 1: plot_tree (built-in)
plt.figure(figsize=(22, 12))
plot_tree(
    dt,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=10,
    proportion=True,
    precision=2,
)
plt.suptitle("Decision Tree Visualization - Iris Dataset", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Method 2: export_text (text representation)
tree_rules = export_text(
    dt, feature_names=feature_names, show_weights=True, decimals=2
)
print("TEXT REPRESENTATION OF THE TREE:")
print(tree_rules)

## Feature Importance

Decision trees provide **feature importance** scores — how much each feature contributes to reducing impurity across all splits.

$$Importance(f_i) = \sum_{nodes} \Delta Impurity \times \frac{N_{node}}{N_{total}}$$

In [ ]:
# Extract and visualize feature importance
importances = dt.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature ranking:")
for i, idx in enumerate(indices):
    print(f"  {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(range(len(importances)), importances[indices], color="steelblue", edgecolor="navy")
axes[0].set_xticks(range(len(importances)))
axes[0].set_xticklabels([feature_names[i] for i in indices], rotation=45, ha="right")
axes[0].set_xlabel("Features", fontsize=12)
axes[0].set_ylabel("Importance", fontsize=12)
axes[0].set_title("Feature Importance - Bar Chart", fontsize=14)

# Horizontal bar chart (sorted)
axes[1].barh(range(len(importances)), importances[indices], color="coral", edgecolor="darkred")
axes[1].set_yticks(range(len(importances)))
axes[1].set_yticklabels([feature_names[i] for i in indices])
axes[1].set_xlabel("Importance", fontsize=12)
axes[1].set_title("Feature Importance - Horizontal", fontsize=14)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Key Hyperparameters

Decision trees are prone to overfitting — they can learn the training data perfectly. Regularization hyperparameters control tree complexity:

| Parameter | Range | Purpose |
|-----------|-------|---------|
| `max_depth` | int or None | Maximum depth of the tree. Deeper trees overfit more. |
| `min_samples_split` | int or float | Minimum samples required to split a node. Higher values → simpler trees. |
| `min_samples_leaf` | int or float | Minimum samples required in a leaf node. Smooths predictions. |
| `max_features` | int, float, or {"sqrt","log2"} | Number of features to consider for each split. Adds randomness (like Random Forest). |
| `max_leaf_nodes` | int or None | Limits the total number of leaf nodes. Alternative to max_depth. |
| `ccp_alpha` | non-negative float | Cost-complexity pruning parameter. Higher → more pruning. |
| `criterion` | {"gini", "entropy", "log_loss"} | Splitting criterion for classification. |
| `splitter` | {"best", "random"} | Strategy for choosing the split at each node. |

### Rule of Thumb
- Start with default parameters, then use **cross-validation** to tune
- A tree with `max_depth=3` to `max_depth=5` is often a good simple model
- Set `min_samples_leaf=5` to prevent leaves with very few samples

In [ ]:
# Systematic study of max_depth on overfitting
depths = range(1, 21)
train_scores = []
test_scores = []

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))

best_depth = depths[np.argmax(test_scores)]
best_test = max(test_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy vs depth
axes[0].plot(depths, train_scores, "bo-", label="Training Accuracy", markersize=6)
axes[0].plot(depths, test_scores, "rs-", label="Testing Accuracy", markersize=6)
axes[0].axvline(best_depth, color="green", linestyle="--", alpha=0.7,
                label=f"Best depth = {best_depth}")
axes[0].set_xlabel("Max Depth", fontsize=12)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title("Effect of max_depth on Overfitting", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].set_xticks(depths[::2])
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0.5, 1.05)

# Right: generalization gap
gap = np.array(train_scores) - np.array(test_scores)
axes[1].plot(depths, gap, "go-", markersize=6)
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Max Depth", fontsize=12)
axes[1].set_ylabel("Training - Testing Accuracy", fontsize=12)
axes[1].set_title("Generalization Gap (Overfitting Signal)", fontsize=14)
axes[1].set_xticks(depths[::2])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best max_depth: {best_depth} (Test Accuracy: {best_test:.4f})")
print(f"At depth=1:  Train={train_scores[0]:.4f}, Test={test_scores[0]:.4f}")
print(f"At depth=20: Train={train_scores[-1]:.4f}, Test={test_scores[-1]:.4f}")
print(f"\nKey insight: As depth increases, training accuracy keeps rising")
print(f"but testing accuracy plateaus or drops — that's overfitting!")

## Pruning: Fighting Overfitting

### Pre-pruning (Early Stopping)
Stop tree growth before it overfits:
- Set `max_depth`, `min_samples_split`, `min_samples_leaf`
- Applied **during** training

### Post-pruning (Cost-Complexity Pruning)
Grow the full tree, then trim branches:
- Uses `ccp_alpha` parameter
- Removes subtrees that don't provide enough improvement
- Scikit-learn provides `cost_complexity_pruning_path()` to find optimal alphas

**Cost-Complexity Formula:**
$$R_\alpha(T) = R(T) + \alpha \times |T|$$
- $R(T)$: Misclassification rate
- $|T|$: Number of leaf nodes
- $\alpha$: Complexity parameter (higher → smaller trees)

In [ ]:
# Cost-complexity pruning
dt_full = DecisionTreeClassifier(random_state=42)
path = dt_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Train trees with different alphas
clfs = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)

train_scores = [c.score(X_train, y_train) for c in clfs]
test_scores = [c.score(X_test, y_test) for c in clfs]
node_counts = [c.tree_.node_count for c in clfs]
depths = [c.tree_.max_depth for c in clfs]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy vs alpha
axes[0, 0].plot(ccp_alphas, train_scores, "bo-", label="Train", markersize=3)
axes[0, 0].plot(ccp_alphas, test_scores, "rs-", label="Test", markersize=3)
axes[0, 0].set_xlabel("ccp_alpha", fontsize=12)
axes[0, 0].set_ylabel("Accuracy", fontsize=12)
axes[0, 0].set_title("Accuracy vs ccp_alpha", fontsize=14)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Node count vs alpha
axes[0, 1].plot(ccp_alphas, node_counts, "go-", markersize=3)
axes[0, 1].set_xlabel("ccp_alpha", fontsize=12)
axes[0, 1].set_ylabel("Number of Nodes", fontsize=12)
axes[0, 1].set_title("Tree Complexity vs ccp_alpha", fontsize=14)
axes[0, 1].grid(alpha=0.3)

# Depth vs alpha
axes[1, 0].plot(ccp_alphas, depths, "mo-", markersize=3)
axes[1, 0].set_xlabel("ccp_alpha", fontsize=12)
axes[1, 0].set_ylabel("Max Depth", fontsize=12)
axes[1, 0].set_title("Tree Depth vs ccp_alpha", fontsize=14)
axes[1, 0].grid(alpha=0.3)

# Best alpha
best_idx = np.argmax(test_scores)
best_alpha = ccp_alphas[best_idx]
best_clf = clfs[best_idx]
axes[1, 1].axis("off")
axes[1, 1].text(
    0.1, 0.5,
    f"Best ccp_alpha: {best_alpha:.4f}\n"
    f"Test Accuracy: {test_scores[best_idx]:.4f}\n"
    f"Tree Nodes: {node_counts[best_idx]}\n"
    f"Tree Depth: {depths[best_idx]}",
    fontsize=13, va="center",
    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8),
)

plt.tight_layout()
plt.show()

print(f"Unpruned tree: {dt_full.tree_.node_count} nodes, depth={dt_full.tree_.max_depth}")
print(f"Pruned tree:   {best_clf.tree_.node_count} nodes, depth={best_clf.tree_.max_depth}")
print(f"Best ccp_alpha: {best_alpha:.6f}")
print(f"Test accuracy improved from {accuracy_score(y_test, dt_full.predict(X_test)):.4f} "
      f"to {test_scores[best_idx]:.4f}")

## Decision Boundary Visualization

For 2D feature space, we can visualize how the tree partitions the space into regions.

In [ ]:
# Use 2 features for visualization: sepal length (0) and petal length (2)
X2 = X[:, [0, 2]]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.3, random_state=42, stratify=y
)

# Train tree with different depths for comparison
depths_to_compare = [2, 4, 10]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

colors_bg = ListedColormap(["#FFCCCC", "#CCFFCC", "#CCCCFF"])
colors_points = ListedColormap(["#FF0000", "#00AA00", "#0000FF"])

for idx, depth in enumerate(depths_to_compare):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X2_train, y2_train)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap=colors_bg)
    axes[idx].contour(xx, yy, Z, colors="black", linewidths=0.5, alpha=0.5)

    for cls in range(3):
        mask_train = y2_train == cls
        mask_test = y2_test == cls
        axes[idx].scatter(
            X2_train[mask_train, 0], X2_train[mask_train, 1],
            c=[colors_points(cls)], edgecolors="k", s=60, label=f"{class_names[cls]} (train)" if idx == 0 else "",
        )
        axes[idx].scatter(
            X2_test[mask_test, 0], X2_test[mask_test, 1],
            c=[colors_points(cls)], edgecolors="k", s=60, marker="s",
        )

    train_acc = clf.score(X2_train, y2_train)
    test_acc = clf.score(X2_test, y2_test)
    axes[idx].set_title(f"max_depth = {depth}\nTrain: {train_acc:.3f} | Test: {test_acc:.3f}", fontsize=12)
    axes[idx].set_xlabel(feature_names[0], fontsize=10)
    axes[idx].set_ylabel(feature_names[2], fontsize=10)
    axes[idx].set_xlim(x_min, x_max)
    axes[idx].set_ylim(y_min, y_max)

axes[0].legend(fontsize=8, loc="upper left")
plt.suptitle("Decision Boundaries at Different Depths", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Shallow trees (depth=2): smooth, high-bias boundaries — may underfit")
print("Deep trees (depth=10): complex, high-variance boundaries — may overfit")

## DecisionTreeRegressor

Decision trees can also handle **regression** tasks. The leaf predicts the **mean value** of the training samples that fall into it.

In [ ]:
# Synthetic regression data
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.1, X_reg.shape[0])
y_reg = y_reg + (X_reg.ravel() - 2.5) ** 2 * 0.3

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

# Compare different depths
depths_reg = [2, 5, None]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

X_plot = np.linspace(X_reg.min(), X_reg.max(), 500).reshape(-1, 1)

for idx, depth in enumerate(depths_reg):
    reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    reg.fit(X_reg_train, y_reg_train)

    y_plot = reg.predict(X_plot)
    y_pred_train = reg.predict(X_reg_train)
    y_pred_test = reg.predict(X_reg_test)

    axes[idx].scatter(X_reg_train, y_reg_train, c="blue", alpha=0.6, s=40, label="Train")
    axes[idx].scatter(X_reg_test, y_reg_test, c="red", alpha=0.6, s=40, label="Test")
    axes[idx].plot(X_plot, y_plot, "g-", linewidth=2.5, label="Prediction")

    mse_train = mean_squared_error(y_reg_train, y_pred_train)
    mse_test = mean_squared_error(y_reg_test, y_pred_test)
    axes[idx].set_title(f"max_depth = {depth}\nMSE Train: {mse_train:.4f} | Test: {mse_test:.4f}", fontsize=12)
    axes[idx].set_xlabel("Feature", fontsize=11)
    axes[idx].set_ylabel("Target", fontsize=11)
    axes[idx].legend(fontsize=9)

plt.suptitle("Decision Tree Regression — Effect of Depth", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## Real-World Example: Wine Dataset

The Wine dataset contains chemical analysis of wines from three cultivars in Italy. Let's see how a decision tree performs on this more challenging multi-class problem.

In [ ]:
# Load Wine dataset
wine = load_wine()
X_w, y_w = wine.data, wine.target
X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_w, y_w, test_size=0.3, random_state=42, stratify=y_w
)

print(f"Wine dataset: {X_w.shape[0]} samples, {X_w.shape[1]} features, {len(np.unique(y_w))} classes")
print(f"Features: {wine.feature_names}")
print(f"Classes: {wine.target_names}")

# Train with best max_depth via CV
dt_wine = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_wine.fit(X_w_train, y_w_train)
y_w_pred = dt_wine.predict(X_w_test)

print(f"\nAccuracy: {accuracy_score(y_w_test, y_w_pred):.4f}")
print("\n" + "="*60)
print("Classification Report:")
print("="*60)
print(classification_report(y_w_test, y_w_pred, target_names=wine.target_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_w_test, y_w_pred))

In [ ]:
# Visualize wine tree
plt.figure(figsize=(24, 12))
plot_tree(
    dt_wine,
    feature_names=wine.feature_names,
    class_names=wine.target_names,
    filled=True,
    rounded=True,
    fontsize=9,
    proportion=True,
    precision=2,
)
plt.suptitle("Decision Tree — Wine Dataset", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for wine
imp_w = dt_wine.feature_importances_
idx_w = np.argsort(imp_w)[::-1]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(imp_w)), imp_w[idx_w], color="teal", edgecolor="darkcyan")
plt.xticks(range(len(imp_w)), [wine.feature_names[i] for i in idx_w], rotation=90, fontsize=10)
plt.xlabel("Features", fontsize=12)
plt.ylabel("Importance", fontsize=12)
plt.title("Feature Importance — Wine Dataset", fontsize=14)

# Annotate top features
for i, (bar, val) in enumerate(zip(bars, imp_w[idx_w])):
    if val > 0.05:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## Comparison with Other Algorithms

How does a single decision tree compare to other common classifiers on the same dataset?

In [ ]:
# Compare multiple algorithms on Iris
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest (100)": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(random_state=42),
    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    results.append({
        "Model": name,
        "Train Acc": f"{train_acc:.4f}",
        "Test Acc": f"{test_acc:.4f}",
        "CV Mean": f"{cv_scores.mean():.4f}",
        "CV Std": f"{cv_scores.std():.4f}",
    })

df_results = pd.DataFrame(results)
print("=" * 80)
print("MODEL COMPARISON ON IRIS DATASET")
print("=" * 80)
print(df_results.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(models))
width = 0.35

train_vals = [float(r["Train Acc"]) for r in results]
test_vals = [float(r["Test Acc"]) for r in results]
cv_vals = [float(r["CV Mean"]) for r in results]
cv_stds = [float(r["CV Std"]) for r in results]

ax.bar(x_pos - width/2, train_vals, width, label="Train Accuracy", color="steelblue", alpha=0.8)
ax.bar(x_pos + width/2, test_vals, width, label="Test Accuracy", color="coral", alpha=0.8)
ax.errorbar(x_pos, cv_vals, yerr=cv_stds, fmt="ko", capsize=5, label="CV Mean ± Std", markersize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels([r["Model"] for r in results], rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Algorithm Comparison on Iris Dataset", fontsize=14)
ax.set_ylim(0.7, 1.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nDecision Tree is interpretable but often less accurate than ensemble methods.")
print("Random Forest usually wins on accuracy but loses interpretability.")

## Bonus: Exporting Tree Rules as Text

The `export_text` function provides a clean text representation — great for reports and documentation.

In [ ]:
# Text tree with depth control
for depth in [2, 3, 4]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    print(f"\n{'='*60}")
    print(f"TREE WITH max_depth = {depth}  (Test Acc: {clf.score(X_test, y_test):.3f})")
    print(f"{'='*60}")
    print(export_text(clf, feature_names=feature_names, spacing=3, decimals=2))

## Pros and Cons of Decision Trees

### ✅ Strengths
| Advantage | Explanation |
|-----------|-------------|
| **Interpretability** | Easy to explain to non-technical stakeholders. A single tree can be visualized. |
| **No scaling needed** | Decision trees are invariant to monotonic transformations. No normalization required. |
| **Handles mixed data** | Can handle both numerical and categorical features (scikit-learn needs numeric). |
| **Non-parametric** | No assumptions about data distribution. |
| **Feature selection** | Built-in feature importance scores. |
| **Handles non-linearity** | Can capture complex non-linear relationships. |

### ❌ Weaknesses
| Disadvantage | Explanation |
|--------------|-------------|
| **High variance / instability** | Small changes in data can produce very different trees. High risk of overfitting. |
| **Bias toward dominant features** | Features with more levels can dominate splits. |
| **Greedy algorithm** | Uses greedy splitting (best split now), which may miss the globally optimal tree. |
| **Poor extrapolation** | Cannot predict values outside the range of training data (regression). |
| **Oblique boundaries** | Splits are axis-aligned (perpendicular to feature axes), which may not fit diagonal patterns well. |

### Mitigation Strategies
1. **Ensemble methods**: Random Forest, Gradient Boosting (bagging + trees)
2. **Pruning**: Cost-complexity pruning to reduce overfitting
3. **Feature selection**: Remove irrelevant features before training
4. **Cross-validation**: Tune hyperparameters systematically

## Practice Exercises

Test your understanding with these exercises:

### Exercise 1: Breast Cancer Classifier
Train a decision tree on the `load_breast_cancer()` dataset. Find the optimal `max_depth` using cross-validation. Print the test accuracy and visualize the best tree.

### Exercise 2: Criterion Comparison
Compare Gini vs Entropy on the **Wine** dataset. Train two trees (one with each criterion) at various depths. Which criterion performs better? Create a side-by-side plot.

### Exercise 3: Tree Structure via Text
Use `export_text` on a decision tree trained on the **Iris** dataset with `max_depth=3`. Read the text rules — can you trace the decision path for a sample with `petal length=3.0` and `petal width=1.5`?

### Exercise 4: Optimal Pruning with GridSearch
Use `GridSearchCV` to find the best `ccp_alpha` along with `max_depth` on the Wine dataset. Report the best parameters and cross-validation score. How does the pruned tree compare to the unpruned one?

### Exercise 5: Decision Boundary on Synthetic Data
Generate 2D synthetic data with `make_classification(n_features=2, n_redundant=0)`. Train a decision tree with `max_depth=None` and plot the decision boundary. Then train one with `max_depth=3` — how do the boundaries differ?